In [ ]:
# Summary: 
# Automate scraping pages of battles from PS, getting their respective replays, 
# and saving them to disk. The 'main' function is toward the end.

In [2]:
# Some basic imports
import numpy as np
import pandas as pd

import json, os, time
import logging, requests
import itertools
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
from urllib.parse import urlencode, urljoin

import re # for regex things

# Useful 'constants'
SEARCH_BASE_URL = "https://replay.pokemonshowdown.com/api/replays/search"
REPLAY_BASE_URL = "https://replay.pokemonshowdown.com/"
USER_BASE_URL = "https://pokemonshowdown.com/users/"

# Only needed if you want messages to appear when an error, etc. occurs
logger = logging.getLogger()
logger.setLevel(logging.ERROR)

# Getting pages/replays
-----------

In [3]:
# Dataclasses
# ===============================================

# Roughly, we 
#    1. Pull pages full of Battles;
#    2. Get `battle_id`s from each Battle;
#    3. Pull and write the Replay using the `battle_id`.
# 
# The `Battle` dataclass is unnecessary at present, but we can use it later if needed.

@dataclass
class Battle:
    uploadtime: int
    id: str
    format: str
    players: list[str]
    rating: Optional[int] = None

@dataclass
class Replay:
    id: str
    format: str
    players: list[str]
    log: str
    inputlog: str
    uploadtime: int
    views: int
    formatid: str
    rating: object = None
    private: int = 0
    password: object = None

In [4]:
# ensuring directory exists
# ===============================================
def ensure_dir(name):
    try:
        Path(name).mkdir(exist_ok=True)
    except OSError as e:
        logger.error("failed to create dir %s: %s", name, e)
        raise SystemExit(1)

In [6]:
# %%%%%%%%% 'Getters' %%%%%%%%%%%%%%%%%%%%%%%%%%%
def get_page(page_num, fmt='') -> list[dict]:
    '''
    Returns a list of dicts, with each dict corresponding to a `json` of a battle.
    The dicts have keys/values like that of the Battle dataclass; `id` is the only crucial one.

    Leaving `fmt=''` searches for battles of any format.
    '''
    
    params = urlencode({"format": fmt, "page": str(page_num)}) 
    url = f"{SEARCH_BASE_URL}?{params}"

    # This deals with an apparent quirk in PS's search: the first-page results are fine, 
    # but subsequent searches seem to want this `Referer` in the GET request.
    headers = {}
    if page_num > 1: 
        headers = {"Referer": f"https://replay.pokemonshowdown.com/?format={fmt}&page={page_num}"}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error("failed to get page %d: %s", page_num, e)
        return []

    # The API response starts with a leading character ']' before valid JSON,
    # so remove it
    raw = response.content[1:]
    
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        logger.error("failed to parse page %d: %s", page_num, e)
        return []
    return data


# ===========================
def get_replay(battle_id: str) -> Replay:
    '''
    Returns a Replay object pulled using the unique battle_id.
    
    Much like `get_page` in function .
    '''
    
    url = REPLAY_BASE_URL + f'{battle_id}' + '.json'
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error("failed to get replay: %s", e)
        return None

    try:
        data = response.json()
    except json.JSONDecodeError as e:
        logger.error("failed to parse replay: %s", e)
        return None

    return Replay(
        id=data.get("id", ""),
        format=data.get("format", ""),
        players=data.get("players", []),
        log=data.get("log", ""),
        inputlog=data.get("inputlog", ""),
        uploadtime=data.get("uploadtime", 0),
        views=data.get("views", 0),
        formatid=data.get("formatid", ""),
        rating=data.get("rating"),
        private=data.get("private", 0),
        password=data.get("password"),
    )

# The 'main' scraper

In [5]:
# Definition
def scrape_replays(page_num, fmt=""):
    '''
    Search page, get battle Replays, and write to file.
    '''
    
    # main page-scraping
    page = get_page(page_num, fmt)
    
    if page == [] : 
        print(f"Failed to get page {page_num}.")
        return None
    
    for battle in page :
        # main replay-scraping
        replay = get_replay(battle['id'])
        if replay is None:
            logger.error("failed to get replay %s", battle['id'])
            return None
        
        # Writing
        if fmt == "" : 
            out_path = Path("replays") / f"{replay.id}.json"
        else : 
            ensure_dir(f'replays/{fmt}')
            out_path = Path(f"replays/{fmt}") / f"{replay.id}.json"
        
        try:
            out_path.write_text(json.dumps(replay.__dict__), encoding="utf-8")
        except OSError as e:
            logger.error("failed to save replay %s : %s", battle['id'], e)

In [ ]:
# Automation/Running

FORMAT='gen9-randombattle';
ensure_dir('./replays')
ensure_dir('./replays/'+FORMAT)

# NOTE: if this value is too small, the PS server (hosted on Cloudflare) 
# will 'cut-off' our scripts (with some unknown severity). Though I have 
# been told that 0.25 seconds 'ought to be fine', this 5 second delay 
# should be plenty fast, and have a wide safety margin.
PAUSE = 5; # seconds between pages

# Note: even trying searching manually, it seems results stop at page 100
for j in range(1,101): 
    print(f"Now working on page {j}.")
    
    scrape_replays(j, fmt=FORMAT)
    
    print(f"\tDone; taking a {PAUSE} second break.")
    time.sleep(PAUSE)

In [7]:
REPLAY_DIR = Path('./replays/gen9-randombattle')

battleid_list = []

for item in REPLAY_DIR.iterdir():
    if (item.is_file() and item.name[0]!='.'): # skipping hidden files
        try:
            with open(REPLAY_DIR/f'{item.name}', 'r') as file:
                x = json.load(file)
                battleid_list.append(x['id'])
        except UnicodeDecodeError as e:
            print(f"{item.name}")
    else : 
        print(f'{item.name}')
        continue

.DS_Store
.ipynb_checkpoints


In [8]:
len(battleid_list)

4798

In [21]:
PAUSE = 1; # seconds between pulls
REPLAY_DIR = Path('./replays/gen9-randombattle')
# Note: even trying searching manually, it seems results stop at page 100
for j in range(4000,len(battleid_list)): 
    
    if j % 20 == 0 : print(f"Now working on replay {j}.")
    # main replay-scraping
    replay = get_replay(battleid_list[j])
    
    if replay is None:
        logger.error("failed to get replay %s", battleid_list[j])
        continue
    
    # Writing
    out_path = REPLAY_DIR / f"{replay.id}.json"
    
    try:
        out_path.write_text(json.dumps(replay.__dict__), encoding="utf-8")
    except OSError as e:
        logger.error("failed to save replay %s : %s", battleid_list[j], e)
    time.sleep(PAUSE)

Now working on replay 4000.
Now working on replay 4020.
Now working on replay 4040.
Now working on replay 4060.
Now working on replay 4080.
Now working on replay 4100.
Now working on replay 4120.
Now working on replay 4140.
Now working on replay 4160.
Now working on replay 4180.
Now working on replay 4200.
Now working on replay 4220.
Now working on replay 4240.
Now working on replay 4260.
Now working on replay 4280.
Now working on replay 4300.
Now working on replay 4320.
Now working on replay 4340.
Now working on replay 4360.
Now working on replay 4380.
Now working on replay 4400.
Now working on replay 4420.
Now working on replay 4440.
Now working on replay 4460.
Now working on replay 4480.
Now working on replay 4500.
Now working on replay 4520.
Now working on replay 4540.
Now working on replay 4560.
Now working on replay 4580.
Now working on replay 4600.
Now working on replay 4620.
Now working on replay 4640.
Now working on replay 4660.
Now working on replay 4680.
Now working on repla